In [3]:
# Q_mat temporal
# channel averaged
# use this!

import numpy as np
import pandas as pd
from numpy.linalg import eigh
import os
from pathlib import Path

In [11]:
def gen_qmat(file_path,save_path):
    if not os.path.exists(save_path):
        print(f'{save_path} is created.')
        os.makedirs(save_path)
        
    # check
    if not os.path.isfile(file_path):
        assert FileNotFoundError


    filename_with_ext = os.path.basename(file_path)
    filename, ext = os.path.splitext(filename_with_ext)

    if ext == '.csv':
        data = pd.read_csv(file_path, header=0)
        data = data.dropna(axis=1, how='all')  # in case there is a column with all nans
        data = data.values
    else:
        data = pd.read_excel(file_path, header=0).values
    train_ratio=0.6 if 'ETT' in file_path else 0.7 # 0.6 for ETT and PEMS
    train_length = int(data.shape[0]*train_ratio)

    for base_ratio in [1.0]:  # base_ratio<1.0 means that only part of trainset is used for computation
        
        ratio = train_ratio * base_ratio

        if ext == '.csv':
            # print(f'Length: {int(data.shape[0]*ratio)}')
            A = data[train_length-int(data.shape[0]*ratio):train_length, 1:].astype(np.float32)
        else:
            # print(f'Length: {int(data.shape[0]*ratio)}')
            A = data[train_length-int(data.shape[0]*ratio):train_length, 0:].astype(np.float32)

        # print(A.shape)

        for time_lag_origin in [24,36,48,60,96,192,336,720]:
            for time_lag_ratio in [8,4,2,1]:
                time_lag = int(time_lag_origin / time_lag_ratio)

                # Initialize a list to store covariance matrices for all features
                Sigma_list = []

                # Loop through all features
                for feature_idx in range(int(A.shape[1]/1)):
                    # Construct the lagged matrix for the current feature
                    lagged_matrix = np.array([
                        A[i:A.shape[0]-time_lag+i+1, feature_idx]
                        for i in range(time_lag)
                    ])  #, dtype=np.float64
                    
                    if np.isnan(lagged_matrix).any():
                        lagged_matrix = np.nan_to_num(lagged_matrix)
                        print('nan in lagged_matrix')
                        
                    # Compute the covariance matrix for the lagged matrix
                    cov_matrix = np.cov(lagged_matrix)
                    diag_vec = np.diag(cov_matrix)
            
                    if (diag_vec < 1e-4).any():
                        continue
                    
                    cov_matrix = cov_matrix / diag_vec  # make sure the diagonal entries are 1

                    Sigma_list.append(np.array(cov_matrix, dtype=np.float32))

                # Average over all features to get the final Sigma
                Sigma = np.mean(Sigma_list, axis=0)


                # Compute eigenvalues and eigenvectors of Sigma
                eigenvalues, eigenvectors = eigh(Sigma)

                q_mat = np.flip(eigenvectors.T, axis=0)

                np.save(os.path.join(save_path, f'{filename}_{time_lag}_ratio{ratio:.1f}.npy'), q_mat)

            # Display the result
            # print(q_mat.shape)
            # print(f'Done! {filename}_{time_lag}_ratio{ratio:.1f}.npy saved')

In [5]:
int(3.8)

3

In [6]:
root_path = "/data/nishome/user1/chaochuan/TSGym_benchmark/dataset"
dataset_dir = [x for x in os.listdir(root_path) if 'ETT' not in x and 'plots_multivariate' not in x]
root_dir = Path(root_path)
file_paths = [str(p) for p in root_dir.rglob('*') if p.is_file() and str(p).endswith('.csv') and "ETT" not in str(p) and 'plots' not in str(p) and 'm4' not in str(p) and '00' not in str(p)]

In [12]:
# ETT
file_paths = ["/data/nishome/user1/chaochuan/TSGym_benchmark/dataset/ETT-small/ETTh1.csv", "/data/nishome/user1/chaochuan/TSGym_benchmark/dataset/ETT-small/ETTh2.csv", "/data/nishome/user1/chaochuan/TSGym_benchmark/dataset/ETT-small/ETTm1.csv", "/data/nishome/user1/chaochuan/TSGym_benchmark/dataset/ETT-small/ETTm2.csv"]

In [13]:
from tqdm import tqdm
for file_path in tqdm(file_paths):
    save_path = str(Path(file_path).parent)
    try:
        gen_qmat(file_path,save_path)
    except Exception as e:
        pass

100%|██████████| 4/4 [00:24<00:00,  6.06s/it]


In [14]:
6144/96

64.0

In [14]:
# csv or excel file
file_path = r'/data/nishome/user1/chaochuan/TSGym_benchmark/dataset/ETT-small/ETTm2.csv'  # replace with csv or excel file;
save_path = "/data/nishome/user1/chaochuan/TSGym_benchmark/dataset/ETT-small/"
if not os.path.exists(save_path):
    print(f'{save_path} is created.')
    os.makedirs(save_path)
    
# check
if not os.path.isfile(file_path):
    assert FileNotFoundError


filename_with_ext = os.path.basename(file_path)
filename, ext = os.path.splitext(filename_with_ext)

if ext == '.csv':
    data = pd.read_csv(file_path, header=0)
    data = data.dropna(axis=1, how='all')  # in case there is a column with all nans
    data = data.values
else:
    data = pd.read_excel(file_path, header=0).values

In [15]:
filename

'ETTm2'

In [16]:
train_ratio=0.6  # 0.6 for ETT and PEMS
train_length = int(data.shape[0]*train_ratio)

for base_ratio in [1.0]:  # base_ratio<1.0 means that only part of trainset is used for computation
    
    ratio = train_ratio * base_ratio

    if ext == '.csv':
        print(f'Length: {int(data.shape[0]*ratio)}')
        A = data[train_length-int(data.shape[0]*ratio):train_length, 1:].astype(np.float32)
    else:
        print(f'Length: {int(data.shape[0]*ratio)}')
        A = data[train_length-int(data.shape[0]*ratio):train_length, 0:].astype(np.float32)

    print(A.shape)

    for time_lag in [96,192,336,720]:

        # Initialize a list to store covariance matrices for all features
        Sigma_list = []

        # Loop through all features
        for feature_idx in range(int(A.shape[1]/1)):
            # Construct the lagged matrix for the current feature
            lagged_matrix = np.array([
                A[i:A.shape[0]-time_lag+i+1, feature_idx]
                for i in range(time_lag)
            ])  #, dtype=np.float64
            
            if np.isnan(lagged_matrix).any():
                lagged_matrix = np.nan_to_num(lagged_matrix)
                print('nan in lagged_matrix')
                
            # Compute the covariance matrix for the lagged matrix
            cov_matrix = np.cov(lagged_matrix)
            diag_vec = np.diag(cov_matrix)
    
            if (diag_vec < 1e-4).any():
                continue
            
            cov_matrix = cov_matrix / diag_vec  # make sure the diagonal entries are 1

            Sigma_list.append(np.array(cov_matrix, dtype=np.float32))

        # Average over all features to get the final Sigma
        Sigma = np.mean(Sigma_list, axis=0)


        # Compute eigenvalues and eigenvectors of Sigma
        eigenvalues, eigenvectors = eigh(Sigma)

        q_mat = np.flip(eigenvectors.T, axis=0)

        np.save(os.path.join(save_path, f'{filename}_{time_lag}_ratio{ratio:.1f}.npy'), q_mat)

        # Display the result
        print(q_mat.shape)
        print(f'Done! {filename}_{time_lag}_ratio{ratio:.1f}.npy saved')

Length: 41808
(41808, 7)
(96, 96)
Done! ETTm2_96_ratio0.6.npy saved
(192, 192)
Done! ETTm2_192_ratio0.6.npy saved
(336, 336)
Done! ETTm2_336_ratio0.6.npy saved
(720, 720)
Done! ETTm2_720_ratio0.6.npy saved


In [7]:
q_mat.shape

(720, 720)

In [ ]:
# 实例化模型
backbone = xLSTMBackbone(
    d_model=256, 
    num_layers=4, 
    max_seq_len=512,
    bidirectional=False # 如果做分类或非生成式任务，建议开启双向
).cuda()

{'verbose': True, 'with_cuda': True, 'extra_ldflags': ['-L/data/nishome/user1/miniconda3/envs/mqenv/lib', '-lcublas'], 'extra_cflags': ['-DSLSTM_HIDDEN_SIZE=256', '-DSLSTM_BATCH_SIZE=8', '-DSLSTM_NUM_HEADS=4', '-DSLSTM_NUM_STATES=4', '-DSLSTM_DTYPE_B=float', '-DSLSTM_DTYPE_R=__nv_bfloat16', '-DSLSTM_DTYPE_W=__nv_bfloat16', '-DSLSTM_DTYPE_G=__nv_bfloat16', '-DSLSTM_DTYPE_S=__nv_bfloat16', '-DSLSTM_DTYPE_A=float', '-DSLSTM_NUM_GATES=4', '-DSLSTM_SIMPLE_AGG=true', '-DSLSTM_GRADIENT_RECURRENT_CLIPVAL_VALID=false', '-DSLSTM_GRADIENT_RECURRENT_CLIPVAL=0.0', '-DSLSTM_FORWARD_CLIPVAL_VALID=false', '-DSLSTM_FORWARD_CLIPVAL=0.0', '-U__CUDA_NO_HALF_OPERATORS__', '-U__CUDA_NO_HALF_CONVERSIONS__', '-U__CUDA_NO_BFLOAT16_OPERATORS__', '-U__CUDA_NO_BFLOAT16_CONVERSIONS__', '-U__CUDA_NO_BFLOAT162_OPERATORS__', '-U__CUDA_NO_BFLOAT162_CONVERSIONS__', '-isystem', '/data/nishome/user1/miniconda3/envs/mqenv/targets/x86_64-linux/include'], 'extra_cuda_cflags': ['-Xptxas="-v"', '-gencode', 'arch=compute_80,co

/data/nishome/user1/miniconda3/envs/mqenv/lib/python3.11/site-packages/xlstm/blocks/slstm/cell.py:543: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @conditional_decorator(
/data/nishome/user1/miniconda3/envs/mqenv/lib/python3.11/site-packages/xlstm/blocks/slstm/cell.py:568: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @conditional_decorator(


{'verbose': True, 'with_cuda': True, 'extra_ldflags': ['-L/data/nishome/user1/miniconda3/envs/mqenv/lib', '-lcublas'], 'extra_cflags': ['-DSLSTM_HIDDEN_SIZE=256', '-DSLSTM_BATCH_SIZE=8', '-DSLSTM_NUM_HEADS=4', '-DSLSTM_NUM_STATES=4', '-DSLSTM_DTYPE_B=float', '-DSLSTM_DTYPE_R=__nv_bfloat16', '-DSLSTM_DTYPE_W=__nv_bfloat16', '-DSLSTM_DTYPE_G=__nv_bfloat16', '-DSLSTM_DTYPE_S=__nv_bfloat16', '-DSLSTM_DTYPE_A=float', '-DSLSTM_NUM_GATES=4', '-DSLSTM_SIMPLE_AGG=true', '-DSLSTM_GRADIENT_RECURRENT_CLIPVAL_VALID=false', '-DSLSTM_GRADIENT_RECURRENT_CLIPVAL=0.0', '-DSLSTM_FORWARD_CLIPVAL_VALID=false', '-DSLSTM_FORWARD_CLIPVAL=0.0', '-U__CUDA_NO_HALF_OPERATORS__', '-U__CUDA_NO_HALF_CONVERSIONS__', '-U__CUDA_NO_BFLOAT16_OPERATORS__', '-U__CUDA_NO_BFLOAT16_CONVERSIONS__', '-U__CUDA_NO_BFLOAT162_OPERATORS__', '-U__CUDA_NO_BFLOAT162_CONVERSIONS__', '-isystem', '/data/nishome/user1/miniconda3/envs/mqenv/targets/x86_64-linux/include'], 'extra_cuda_cflags': ['-Xptxas="-v"', '-gencode', 'arch=compute_80,co

In [ ]:
# 模拟输入 (Batch=32, SeqLen=100, DModel=256)
x = torch.randn(32, 100, 256).cuda()

# 前向传播
y = backbone(x)

print(y.shape) 
# 输出: torch.Size([32, 100, 256])